In [2]:
import geopandas as gpd
import gcsfs
import google.auth
import pandas as pd

import world_cup_vars as wc_vars

credentials, _ = google.auth.default()

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

In [3]:
routes_near_stadium = pd.read_parquet(
    f"{GCS_FILE_PATH}routes_near_poi.parquet", filesystem=gcsfs.GCSFileSystem()
)

In [4]:
route_gdf = pd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_schedule_rt_route_direction_summary_world_cup.parquet", 
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["service_date", "schedule_name", "feed_key", "route_id", "route_id_cleaned",
               "route_name", "direction_id", "route_type",
               "shape_id", "shape_array_key", 
               "n_trips", "n_shapes", "num_stop_times", "avg_stops_served"
              ],
    filters = [[
        ("schedule_name", "in", wc_vars.socal_names + wc_vars.bay_area_names),
    ]]
).merge(
    routes_near_stadium,
    on = ["schedule_name", "route_name", "direction_id", "shape_array_key"],
    how = "inner"
)

In [8]:
for i in sorted(route_gdf.schedule_name.unique()):
    print(i)
    print(route_gdf[route_gdf.schedule_name == i].route_name.value_counts())
    print("--------------------------------------------")

BART Schedule
route_name
5__Green-S Berryessa/North San Jose to Daly City    29
6__Green-N Daly City to Berryessa/North San Jose    29
3__Orange-N Berryessa/North San Jose to Richmond    29
4__Orange-S Richmond to Berryessa/North San Jose    29
Name: count, dtype: int64
--------------------------------------------
Bay Area 511 ACE Schedule
route_name
ACE__Altamont Commuter Express    29
Name: count, dtype: int64
--------------------------------------------
Bay Area 511 BART Schedule
route_name
Green__Green-S Berryessa/North San Jose to Daly City     22
Green__Green-N Daly City to Berryessa/North San Jose     22
Orange__Orange-N Berryessa/North San Jose to Richmond    22
Orange__Orange-S Richmond to Berryessa/North San Jose    22
Name: count, dtype: int64
--------------------------------------------
Bay Area 511 Caltrain Schedule
route_name
Limited__Limited                                                 34
Express__Express                                                 34
Local Weekda

In [30]:
import polars as pl
from great_tables import GT

one_operator = "Big Blue Bus Schedule"
operator_df = route_gdf[route_gdf.schedule_name == one_operator]

operator_wide_df = (
    operator_df
    .sort_values(["schedule_name", "route_name", "direction_id", "service_date"])
    .groupby(["schedule_name", "route_name", "direction_id"])
    .agg({
        "service_date": lambda x: list(pd.to_datetime(x).dt.date),
        "n_trips": lambda x: list(x),
        "num_stop_times": lambda x: list(x),
        "avg_stops_served": lambda x: list(x)
    }).reset_index()
)

(
    GT(pl.from_pandas(operator_wide_df.drop(columns = "service_date")))
    .fmt_nanoplot("n_trips")
    .fmt_nanoplot("num_stop_times")
    .fmt_nanoplot("avg_stops_served")
)

GT(_tbl_data=shape: (4, 6)
┌────────────────┬────────────────┬──────────────┬────────────────┬────────────────┬───────────────┐
│ schedule_name  ┆ route_name     ┆ direction_id ┆ n_trips        ┆ num_stop_times ┆ avg_stops_ser │
│ ---            ┆ ---            ┆ ---          ┆ ---            ┆ ---            ┆ ved           │
│ str            ┆ str            ┆ i64          ┆ list[i64]      ┆ list[i64]      ┆ ---           │
│                ┆                ┆              ┆                ┆                ┆ list[f64]     │
╞════════════════╪════════════════╪══════════════╪════════════════╪════════════════╪═══════════════╡
│ Big Blue Bus   ┆ 3__3 Lincoln   ┆ 0            ┆ [95, 66, … 95] ┆ [6768, 4464, … ┆ [36.0, 36.0,  │
│ Schedule       ┆ Boulevard/LAX  ┆              ┆                ┆ 6840]          ┆ … 36.0]       │
│ Big Blue Bus   ┆ 3__3 Lincoln   ┆ 1            ┆ [95, 67, … 95] ┆ [6171, 4125, … ┆ [33.0, 33.0,  │
│ Schedule       ┆ Boulevard/LAX  ┆              ┆                ┆ 6270]          ┆ … 33.0]       │
│ Big Blue Bus   ┆ T14__T14 Los   ┆ 0            ┆ [14, 14, … 14] ┆ [56, 52, … 56] ┆ [2.0, 2.0, …  │
│ Schedule       ┆ Angeles        ┆              ┆                ┆                ┆ 2.0]          │
│                ┆ Stadium        ┆              ┆                ┆                ┆               │
│ Big Blue Bus   ┆ T14__T14 Los   ┆ 1            ┆ [12, 12, … 12] ┆ [44, 32, … 46] ┆ [2.0, 2.0, …  │
│ Schedule       ┆ Angeles        ┆              ┆                ┆                ┆ 2.0]          │
│                ┆ Stadium        ┆              ┆                ┆                ┆               │
└────────────────┴────────────────┴──────────────┴────────────────┴────────────────┴───────────────┘, _body=<great_tables._gt_data.Body object at 0x7cc7ce523290>, _boxhead=Boxhead([ColInfo(var='schedule_name', type=<ColInfoTypeEnum.default: 1>, column_label='schedule_name', column_align='left', column_width=None), ColInfo(var='route_name', type=<ColInfoTypeEnum.default: 1>, column_label='route_name', column_align='left', column_width=None), ColInfo(var='direction_id', type=<ColInfoTypeEnum.default: 1>, column_label='direction_id', column_align='right', column_width=None), ColInfo(var='n_trips', type=<ColInfoTypeEnum.default: 1>, column_label='n_trips', column_align='center', column_width=None), ColInfo(var='num_stop_times', type=<ColInfoTypeEnum.default: 1>, column_label='num_stop_times', column_align='center', column_width=None), ColInfo(var='avg_stops_served', type=<ColInfoTypeEnum.default: 1>, column_label='avg_stops_served', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7cc7ce6d9c10>, _spanners=Spanners([]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7cc7caa19a90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7cc7cb571050>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7cc7cabfb610>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7cc7d9649dd0>, <great_tables._gt_data.FormatInfo object at 0x7cc7cacc5e10>, <great_tables._gt_data.FormatInfo object at 0x7cc7caa0bb90>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[])

In [12]:
operator_df.dtypes

service_date         datetime64[ns]
schedule_name                object
feed_key                     object
route_id                     object
route_id_cleaned             object
route_name                   object
direction_id                  int64
route_type                   object
shape_id                     object
shape_array_key              object
n_trips                       int64
n_shapes                      int64
num_stop_times                int64
avg_stops_served            float64
point_of_interest            object
dtype: object

In [19]:
route_gdf.route_type.unique()

array(['3', '2', '1', '0'], dtype=object)

In [3]:
route_feeds = (
    route_gdf
    .groupby(["schedule_name"])
    .agg({"feed_key": "nunique"})
    .reset_index()
)

In [4]:
subset_feeds = pd.read_parquet(
    f"{GCS_FILE_PATH}feeds_world_cup.parquet", 
    filesystem=gcsfs.GCSFileSystem()
)

subset_feeds_counts = (
    subset_feeds
    .groupby("gtfs_dataset_name")
    .agg({"feed_key": "nunique"})
    .reset_index()
    .rename(columns = {"gtfs_dataset_name": "schedule_name"})
)

In [5]:
m1 = pd.merge(
    route_feeds, 
    subset_feeds_counts,
    on = "schedule_name",
    how = "outer",
    indicator=True
)

m1._merge.value_counts()

_merge
both          18
right_only     5
left_only      0
Name: count, dtype: int64

In [6]:
m1[m1._merge== "right_only"]

,schedule_name,feed_key_x,feed_key_y,_merge
0,ACE Schedule,NaN,1,right_only
1,Amtrak Schedule,NaN,28,right_only
10,Caltrain Schedule,NaN,1,right_only
11,Capitol Corridor Schedule,NaN,3,right_only
14,Inglewood Schedule,NaN,2,right_only


In [7]:
m1[(m1._merge=="both") & (m1.feed_key_x != m1.feed_key_y)]

,schedule_name,feed_key_x,feed_key_y,_merge
4,Bay Area 511 BART Schedule,1.0,2,both
15,LA DOT Schedule,11.0,12,both
17,LA Metro Events Schedule,1.0,3,both
18,LA Metro Rail Schedule,19.0,20,both


In [8]:
route_gdf.feed_key.nunique() # this is ok, overcount a couple, because filtering is by name

51

In [9]:
subset_feeds.feed_key.nunique()

91

In [10]:
shape_geom = gpd.read_parquet(
    f"{GCS_FILE_PATH}dim_shape_arrays_world_cup.parquet",
    storage_options = {"token": credentials},
    #columns = ["shape_array_key", "geometry"]
)

In [11]:
shape_geom.shape

(7568, 4)

In [12]:
shape_geom.feed_key.nunique()

87

In [13]:
shape_geom.shape_array_key.nunique()

7568

In [14]:
# Ok, this is just a couple of feeds that were missing before
gdf = pd.merge(
    route_gdf,
    shape_geom,
    on = ["feed_key", "shape_id"],
    how = "left",
    indicator=True
)

gdf._merge.value_counts()

_merge
both          19085
left_only         0
right_only        0
Name: count, dtype: int64

In [16]:
gdf.route_type

AttributeError: 'DataFrame' object has no attribute 'route_type'

In [15]:
gdf = pd.merge(
    route_gdf,
    shape_geom,
    on = "shape_array_key",
    how = "left",
    indicator=True
)

gdf._merge.value_counts()

_merge
both          19085
left_only         0
right_only        0
Name: count, dtype: int64

In [ ]:
def merge_routes_with_shape_geom(
    route_df: pd.DataFrame,
):
    shape_geom = gpd.read_parquet(
        f"{GCS_FILE_PATH}dim_shape_arrays_world_cup.parquet",
        storage_options = {"token": credentials},
        columns = ["feed_key", "shape_id", "shape_array_key", "geometry"]
    )
    
    gdf = pd.merge(
        route_df,
        shape_geom,
        on = ["feed_key", "shape_id"],
        how = "left",
        indicator=True
    )

    gdf = gpd.GeoDataFrame(gdf, geometry = "geometry")
    
    return gdf

In [ ]:
route_gdf2 = route_gdf.pipe(merge_routes_with_shape_geom)

In [ ]:
route_gdf.shape, route_gdf2.shape

In [ ]:
route_gdf2._merge.value_counts()

## filter for operators, then filter for routes

filtering for routes is really time-consuming

can we just do geospatial pass, grab all the `routes` where shapes get within 3 miles or 10 miles of stadium?

In [ ]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}daily_schedule_rt_route_direction_summary_combined_dates.parquet",
    filesystem = gcsfs.GCSFileSystem(),
)

FutureWarning - `isin` behavior

/tmp/ipykernel_2595/3934463677.py:1: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

In [ ]:
sofi_dates = [pd.to_datetime(d) for d in wc_vars.sofi_dates]
levi_dates = [pd.to_datetime(d) for d in wc_vars.levi_dates]

In [ ]:
# isin needs to cast to match dtypes, got 
df_socal = df[(df.service_date.isin(sofi_dates)) & (df.schedule_name.isin(wc_vars.socal_names))]
df_norcal = df[(df.service_date.isin(levi_dates)) & (df.schedule_name.isin(wc_vars.bay_area_names))]

In [ ]:
df = pd.concat([df_socal, df_norcal], axis=0, ignore_index=True)

In [ ]:
la_metro_routes = [
    '22__South Bay Dodger Stadium Express',
    '803__ Metro C Line',
    '807__ Metro K Line'
]

ladot_routes = [
    'CE' # commuter express, so many, need to filter these down more
]

In [ ]:
# filter for routes
df[
    (df.schedule_name == "LA DOT Schedule") & 
    (df.route_name.str.contains("CE"))
    ].route_name.value_counts()